
# Single-affinity theory — phase diagrams

This notebook is dedicated to the **state/resource geometry** of the revised single-affinity theory.

The central state-local objects are

\[
\chi_{h,\gamma}(x,b),\qquad
T_\pi(x,b),\qquad
\eta_{\rm IR}(x,b),
\]

and the ensemble-level thermodynamic objects are

\[
J_c,\quad I_{\rm sens},\quad
\Delta S_{\rm sys},\quad
\Sigma,\quad
C_{\rm th}=hJ_c+I_{\rm sens},\quad
\eta_{\rm th}=\frac{hJ_c}{hJ_c+I_{\rm sens}}.
\]

The notebook has three complementary "phase spaces":

1. **state × actuation:** \((x,b)\),
2. **ensemble center × actuation:** \((x_0,b)\),
3. **sensing × actuation resources:** \((q_c,b)\).

All theory is imported from `theory_revised.py`; the notebook only organizes parameter sweeps and plots.


In [ ]:

from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

from theory_revised import (
    TheoryParameters,
    single_affinity_reference,
    binomial_ensemble,
    finite_horizon_thermodynamics,
    calibrate_affinity_compliance_from_counts,
)

print("Loaded theory_revised.py")


## 1. Baseline parameters

In [ ]:

N = 24
q_c = 12
beta = 4.0
theta = 0.5
h = 2.0
gamma = 0.75

x0_baseline = 0.25
b_slices = [1, 6, 12, 18, 24]


In [ ]:

def make_reference(*, b, q_c=q_c, h=h, gamma=gamma,
                   beta=beta, theta=theta, N=N):
    return single_affinity_reference(
        TheoryParameters(
            N=int(N),
            q_c=int(q_c),
            b=int(b),
            beta=float(beta),
            theta=float(theta),
            h=float(h),
            gamma=float(gamma),
        )
    )


def heatmap(values, x_values, y_values, title, cbar_label,
            xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(8.2, 5.4))
    im = ax.imshow(
        values,
        origin="lower",
        aspect="auto",
        extent=[x_values[0], x_values[-1], y_values[0], y_values[-1]],
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(cbar_label)
    plt.tight_layout()
    plt.show()


def build_x_b_landscape(*, q_c=q_c, h=h, gamma=gamma,
                        beta=beta, theta=theta, N=N):
    b_values = np.arange(0, N + 1, dtype=int)
    x_values = np.arange(0, N + 1, dtype=float) / N

    chi = np.zeros((len(b_values), len(x_values)))
    T_pi = np.zeros_like(chi)
    eta_ir = np.full_like(chi, np.nan)
    advocacy = np.zeros_like(chi)
    pinsker = np.zeros_like(chi)
    entropy_ceiling = np.zeros_like(chi)

    for i, b in enumerate(b_values):
        ref = make_reference(
            b=b, q_c=q_c, h=h, gamma=gamma,
            beta=beta, theta=theta, N=N
        )
        chi[i] = ref.chi
        T_pi[i] = ref.T_pi
        eta_ir[i] = ref.eta_IR
        advocacy[i] = ref.advocacy
        pinsker[i] = ref.pinsker_bound
        entropy_ceiling[i] = ref.entropy_ceiling()

    return {
        "b": b_values,
        "x": x_values,
        "chi": chi,
        "T_pi": T_pi,
        "eta_ir": eta_ir,
        "advocacy": advocacy,
        "pinsker": pinsker,
        "entropy_ceiling": entropy_ceiling,
    }


def line_slices(quantity_name, ylabel, *, b_values=b_slices,
                q_c=q_c, h=h, gamma=gamma, beta=beta,
                theta=theta, N=N):
    fig, ax = plt.subplots(figsize=(8, 5))
    for b in b_values:
        ref = make_reference(
            b=b, q_c=q_c, h=h, gamma=gamma,
            beta=beta, theta=theta, N=N
        )
        ax.plot(
            ref.shares,
            getattr(ref, quantity_name),
            marker="o",
            markersize=3,
            label=f"b={b}",
        )
    ax.set_xlabel("Current target fraction x = n/N")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()



# Part I — State-local \(x\times b\) phase diagrams

These are the diagrams most directly comparable to a phase-space view of the controller.


In [ ]:
local = build_x_b_landscape()


## 2. Susceptibility \(\chi(x,b)\)

\[
\chi_{h,\gamma}(x,b)
=
[\sigma(h)-x]
\left[
1-\left(1-\frac{\gamma}{N}\right)^b
\right].
\]

This surface shows the set point \(x^\star=\sigma(h)\), the growth of response with \(b\), and diminishing actuation returns.


In [ ]:

heatmap(
    local["chi"], local["x"], local["b"],
    r"Susceptibility phase diagram: $\chi_{h,\gamma}(x,b)$",
    r"$\chi$ (target-fraction response)",
    "Current target fraction x = n/N", "Actuation budget b"
)


In [ ]:

line_slices("chi", r"$\chi_{h,\gamma}(x)$")



## 3. Action-to-population information \(T_\pi(x,b)\)

\[
T_\pi(n)=I(U_k;n_{k+1}\mid n_k=n).
\]

High values require both action variability and separation between the ADVOCATE and NoOp kernels.


In [ ]:

heatmap(
    local["T_pi"], local["x"], local["b"],
    r"Action-to-population information: $T_\pi(x,b)$",
    r"$T_\pi$ [bits]",
    "Current target fraction x = n/N", "Actuation budget b"
)


In [ ]:

line_slices("T_pi", r"$T_\pi(n)$ [bits]")



## 4. Information-response efficiency \(\eta_{\rm IR}(x,b)\)

\[
\eta_{\rm IR}(n)
=
\frac{2a_n(1-a_n)\chi(n)^2}
{(\ln2)\,T_\pi(n)}
\le 1.
\]


In [ ]:

heatmap(
    local["eta_ir"], local["x"], local["b"],
    r"Information-response efficiency: $\eta_{\rm IR}(x,b)$",
    r"$\eta_{\rm IR}$",
    "Current target fraction x = n/N", "Actuation budget b"
)


In [ ]:

line_slices("eta_IR", r"$\eta_{\rm IR}(n)$")



## 5. Information geometry: ceiling and response lower bound

\[
T_\pi(n)\le H_2(a_n),
\qquad
T_\pi(n)\ge
\frac{2a_n(1-a_n)}{\ln2}\chi(n)^2.
\]


In [ ]:

heatmap(
    local["entropy_ceiling"], local["x"], local["b"],
    r"Action-entropy ceiling $H_2(a_n)$",
    "bits",
    "Current target fraction x = n/N", "Actuation budget b"
)

heatmap(
    local["pinsker"], local["x"], local["b"],
    r"Pinsker response lower bound",
    "bits",
    "Current target fraction x = n/N", "Actuation budget b"
)



# Part II — Thermodynamic \(x_0\times b\) phase diagrams

Thermodynamic quantities require an occupancy distribution. We use

\[
p_0(n)=\mathrm{Binomial}(N,x_0)
\]

as an experimentally convenient family.

For large sweeps, the exact decomposition is evaluated with the direct path-KL check disabled; a representative point is checked separately below.


In [ ]:

def build_x0_b_thermo_landscape(
    *, q_c=q_c, h=h, gamma=gamma, beta=beta,
    theta=theta, N=N, x0_values=None
):
    if x0_values is None:
        x0_values = np.linspace(0.02, 0.98, 49)

    b_values = np.arange(0, N + 1, dtype=int)

    eta_th = np.full((len(b_values), len(x0_values)), np.nan)
    eta_th_signed = np.full_like(eta_th, np.nan)
    valid = np.zeros_like(eta_th, dtype=bool)
    J_c = np.zeros_like(eta_th)
    I_sens = np.zeros_like(eta_th)
    C_th = np.zeros_like(eta_th)
    Sigma = np.zeros_like(eta_th)
    delta_S = np.zeros_like(eta_th)

    for i, b in enumerate(b_values):
        ref = make_reference(
            b=b, q_c=q_c, h=h, gamma=gamma,
            beta=beta, theta=theta, N=N
        )
        for j, x0 in enumerate(x0_values):
            p0 = binomial_ensemble(N, float(x0))
            cycle = ref.one_cycle(p0, check_kl=False)

            eta_th_signed[i, j] = cycle.eta_th
            valid[i, j] = cycle.eta_th_has_bounded_interpretation
            if cycle.eta_th_has_bounded_interpretation:
                eta_th[i, j] = cycle.eta_th

            J_c[i, j] = cycle.J_c
            I_sens[i, j] = cycle.I_sens_nats
            C_th[i, j] = cycle.C_th_nats
            Sigma[i, j] = cycle.Sigma_nats
            delta_S[i, j] = cycle.delta_S_sys_nats

    return {
        "b": b_values,
        "x0": np.asarray(x0_values),
        "eta_th": eta_th,
        "eta_th_signed": eta_th_signed,
        "valid": valid,
        "J_c": J_c,
        "I_sens": I_sens,
        "C_th": C_th,
        "Sigma": Sigma,
        "delta_S": delta_S,
    }


thermo = build_x0_b_thermo_landscape()



## 6. Thermodynamic efficiency \(\eta_{\rm th}(x_0,b)\)

For target-directed operation,

\[
\eta_{\rm th}
=
\frac{hJ_c}{hJ_c+I_{\rm sens}}
\le1.
\]

Cells outside the target-directed regime are masked rather than clipped.


In [ ]:

heatmap(
    thermo["eta_th"], thermo["x0"], thermo["b"],
    r"Thermodynamic efficiency: $\eta_{\rm th}(x_0,b)$",
    r"$\eta_{\rm th}$",
    r"Binomial ensemble center $x_0$", "Actuation budget b"
)



## 7. Controlled current \(J_c(x_0,b)\)

\[
J_c=N\sum_n p(n)a_n\chi(n).
\]


In [ ]:

heatmap(
    thermo["J_c"], thermo["x0"], thermo["b"],
    r"Mean controlled current: $J_c(x_0,b)$",
    r"$J_c$ [target-count change / cycle]",
    r"Binomial ensemble center $x_0$", "Actuation budget b"
)



## 8. Sensing, non-storage expenditure, irreversibility, and storage

\[
C_{\rm th}=hJ_c+I_{\rm sens},
\qquad
\Sigma=\Delta S_{\rm sys}+C_{\rm th}.
\]


In [ ]:

heatmap(
    thermo["I_sens"], thermo["x0"], thermo["b"],
    r"Sensing information $I_{\rm sens}(x_0,b)$",
    "nats",
    r"Binomial ensemble center $x_0$", "Actuation budget b"
)


In [ ]:

heatmap(
    thermo["C_th"], thermo["x0"], thermo["b"],
    r"Non-storage expenditure $C_{\rm th}(x_0,b)$",
    "nats",
    r"Binomial ensemble center $x_0$", "Actuation budget b"
)


In [ ]:

heatmap(
    thermo["Sigma"], thermo["x0"], thermo["b"],
    r"Finite-time irreversibility $\Sigma(x_0,b)$",
    "nats",
    r"Binomial ensemble center $x_0$", "Actuation budget b"
)


In [ ]:

heatmap(
    thermo["delta_S"], thermo["x0"], thermo["b"],
    r"Transient storage $\Delta S_{\rm sys}(x_0,b)$",
    "nats",
    r"Binomial ensemble center $x_0$", "Actuation budget b"
)



## 9. Direct KL validation at a representative point

The baseline report point admits a direct numerical check of

\[
\Delta S_{\rm sys}+hJ_c+I_{\rm sens}=\Sigma.
\]


In [ ]:

validation_ref = make_reference(b=12)
validation_p0 = binomial_ensemble(N, 0.25)
validation_cycle = validation_ref.one_cycle(validation_p0, check_kl=True)

print("Delta S_sys =", validation_cycle.delta_S_sys_nats)
print("h J_c       =", h * validation_cycle.J_c)
print("I_sens      =", validation_cycle.I_sens_nats)
print("Sigma       =", validation_cycle.Sigma_nats)
print("direct KL   =", validation_cycle.Sigma_direct_KL_nats)
print("residual    =", validation_cycle.identity_residual_nats)
print("eta_th      =", validation_cycle.eta_th)



# Part III — Sensing × actuation resource phase diagrams

At fixed initial occupancy, vary sensing and actuation independently in the \((q_c,b)\) plane.


In [ ]:

def build_qc_b_resource_landscape(
    *, x0=x0_baseline, h=h, gamma=gamma, beta=beta,
    theta=theta, N=N, qc_values=None
):
    if qc_values is None:
        qc_values = np.arange(1, N + 1, dtype=int)
    b_values = np.arange(0, N + 1, dtype=int)
    p0 = binomial_ensemble(N, x0)

    eta_th = np.full((len(b_values), len(qc_values)), np.nan)
    J_c = np.zeros_like(eta_th)
    I_sens = np.zeros_like(eta_th)

    for i, b in enumerate(b_values):
        for j, qc in enumerate(qc_values):
            ref = make_reference(
                b=b, q_c=int(qc), h=h, gamma=gamma,
                beta=beta, theta=theta, N=N
            )
            cycle = ref.one_cycle(p0, check_kl=False)

            if cycle.eta_th_has_bounded_interpretation:
                eta_th[i, j] = cycle.eta_th
            J_c[i, j] = cycle.J_c
            I_sens[i, j] = cycle.I_sens_nats

    return {
        "b": b_values,
        "q_c": np.asarray(qc_values),
        "eta_th": eta_th,
        "J_c": J_c,
        "I_sens": I_sens,
    }


resources = build_qc_b_resource_landscape()


In [ ]:

heatmap(
    resources["eta_th"], resources["q_c"], resources["b"],
    rf"Sensing-actuation efficiency plane at $x_0={x0_baseline}$",
    r"$\eta_{\rm th}$",
    r"Sensing budget $q_c$", "Actuation budget b"
)


In [ ]:

heatmap(
    resources["J_c"], resources["q_c"], resources["b"],
    rf"Controlled current in resource space at $x_0={x0_baseline}$",
    r"$J_c$",
    r"Sensing budget $q_c$", "Actuation budget b"
)


In [ ]:

heatmap(
    resources["I_sens"], resources["q_c"], resources["b"],
    rf"Sensing information in resource space at $x_0={x0_baseline}$",
    "nats",
    r"Sensing budget $q_c$", "Actuation budget b"
)



# Part IV — Parameter experiments

Use `explore_theory(...)` to regenerate the three central state-local phase diagrams after changing \(h,\gamma,q_c,\beta,\theta\).


In [ ]:

def explore_theory(*, N=24, q_c=12, beta=4.0, theta=0.5,
                   h=2.0, gamma=0.75):
    data = build_x_b_landscape(
        N=N, q_c=q_c, beta=beta, theta=theta, h=h, gamma=gamma
    )

    heatmap(
        data["chi"], data["x"], data["b"],
        rf"$\chi(x,b)$ | $h={h}$, $\gamma={gamma}$, $q_c={q_c}$",
        r"$\chi$", "Current target fraction x", "Actuation budget b"
    )
    heatmap(
        data["T_pi"], data["x"], data["b"],
        rf"$T_\pi(x,b)$ | $h={h}$, $\gamma={gamma}$, $q_c={q_c}$",
        r"$T_\pi$ [bits]", "Current target fraction x", "Actuation budget b"
    )
    heatmap(
        data["eta_ir"], data["x"], data["b"],
        rf"$\eta_{{\rm IR}}(x,b)$ | $h={h}$, $\gamma={gamma}$, $q_c={q_c}$",
        r"$\eta_{\rm IR}$", "Current target fraction x", "Actuation budget b"
    )
    return data


### Feedback-gain comparison

In [ ]:

beta_1 = build_x_b_landscape(beta=1.0)
beta_8 = build_x_b_landscape(beta=8.0)

heatmap(
    beta_1["T_pi"], beta_1["x"], beta_1["b"],
    r"$T_\pi(x,b)$ for $\beta=1$",
    "bits", "Current target fraction x", "Actuation budget b"
)

heatmap(
    beta_8["T_pi"], beta_8["x"], beta_8["b"],
    r"$T_\pi(x,b)$ for $\beta=8$",
    "bits", "Current target fraction x", "Actuation budget b"
)


### Kinetic-compliance comparison

In [ ]:

gamma_low = build_x_b_landscape(gamma=0.25)
gamma_high = build_x_b_landscape(gamma=1.0)

heatmap(
    gamma_low["chi"], gamma_low["x"], gamma_low["b"],
    r"$\chi(x,b)$ for $\gamma=0.25$",
    r"$\chi$", "Current target fraction x", "Actuation budget b"
)

heatmap(
    gamma_high["chi"], gamma_high["x"], gamma_high["b"],
    r"$\chi(x,b)$ for $\gamma=1$",
    r"$\chi$", "Current target fraction x", "Actuation budget b"
)



# Part V — Calibrated LLM regime

Using the microscopic controlled-transition counts from the susceptibility study:

\[
\gamma_{\rm eff}=p_++p_-,
\qquad
h_{\rm eff}=\log\frac{p_+}{p_-}.
\]


In [ ]:

calibration = calibrate_affinity_compliance_from_counts(
    plus_transitions=208,
    plus_eligible=572,
    minus_transitions=4,
    minus_eligible=508,
)

print("h_eff     =", calibration.h_eff)
print("gamma_eff =", calibration.gamma_eff)
print("odds       =", calibration.forward_reverse_odds)


In [ ]:

calibrated = explore_theory(
    N=24,
    q_c=12,
    beta=4.0,
    theta=0.5,
    h=calibration.h_eff,
    gamma=calibration.gamma_eff,
)


# Part VI — Finite-horizon thermodynamics

In [ ]:

ref = make_reference(b=12)
p0 = binomial_ensemble(N, x0_baseline)

for H in [1, 2, 5, 10, 20]:
    result = finite_horizon_thermodynamics(
        ref, p0, rounds=H, check_kl=False
    )
    print(
        f"H={H:2d} | "
        f"J_c={result.total_J_c: .5f} | "
        f"I_sens={result.total_I_sens_nats: .5f} | "
        f"Sigma={result.total_Sigma_nats: .5f} | "
        f"eta_th={result.eta_th: .6f}"
    )



## Suggested experiments

- \(h\in\{0.5,1,2,4\}\): move the intrinsic controller set point.
- \(\gamma\in\{0.1,0.25,0.5,0.75,1\}\): change kinetic compliance.
- \(\beta\in\{1,2,4,8\}\): sharpen the feedback decision boundary.
- \(\theta\in\{0.25,0.4,0.5,0.75\}\): move the switching region.
- \(q_c\in\{6,12,18\}\): compare sensing-resource regimes.
- Compare the baseline theory with the calibrated \(h_{\rm eff},\gamma_{\rm eff}\) regime.

The most important theory/data-comparison surfaces are

\[
\chi(x,b),\qquad
T_\pi(x,b),\qquad
\eta_{\rm IR}(x,b),
\]

followed by

\[
\eta_{\rm th}(x_0,b)
\quad\text{and}\quad
\eta_{\rm th}(q_c,b).
\]
